# RAG Assignment 2 Code Notebook
Domain: DJI drone/product documentation and drone regulation/product-support questions.

This notebook implements:
1. Baseline RAG pipeline: chunking → embedding → FAISS vector store → simple vector retrieval → vanilla prompt → answer
2. Enhanced RAG pipeline: chunking → embedding → FAISS vector store → hybrid retrieval → explicit reranking → grounded prompt → answer
3. Retrieval and generation evaluation
4. Demo log export

Before running:
- Upload `the-drone-code-march-2026.pdf`
- Upload `DJI_Product.zip`


# A. Preparation & upload

In [ ]:
# 0. Install dependencies
# ============================================================

!pip -q install pymupdf python-docx beautifulsoup4 lxml pandas numpy tqdm scikit-learn
!pip -q install sentence-transformers faiss-cpu rank-bm25 google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 53.4 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Install local generation dependencies
# ============================================================

!pip install -q transformers accelerate torch

In [ ]:
# ============================================================
# 1. Upload data / prepare data folder
# ============================================================

from pathlib import Path
import shutil
import zipfile

# Use the current working directory as the project base directory
BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "rag_data"
EXTRACT_DIR = BASE_DIR / "rag_extracted"
OUTPUT_DIR = BASE_DIR / "rag_outputs"

DATA_DIR.mkdir(exist_ok=True)
EXTRACT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Detect whether the notebook is running in Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ZIP_NAME = "drone.zip"
ZIP_PATH = BASE_DIR / ZIP_NAME

if IN_COLAB:
    print("Running in Google Colab.")
    print("Please upload drone.zip.")

    uploaded = files.upload()

    for filename in uploaded.keys():
        src = Path(filename)
        dst = BASE_DIR / filename

        # If a file with the same name already exists, replace it
        if dst.exists() and src.resolve() != dst.resolve():
            dst.unlink()

        shutil.move(str(src), str(dst))
        print(f"Uploaded: {filename} -> {dst}")

else:
    print("Running outside Google Colab.")
    print(f"Please make sure {ZIP_NAME} is placed in: {BASE_DIR}")

# Check whether drone.zip exists
if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"{ZIP_NAME} was not found. Please upload it in Colab or place it in the project folder."
    )

# Extract drone.zip into DATA_DIR
with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR)

print("\nData extracted successfully.")
print("Current files in DATA_DIR:")

for p in DATA_DIR.rglob("*"):
    if p.is_file():
        print("-", p.relative_to(DATA_DIR))

Running in Google Colab.
Please upload drone.zip.


Saving drone.zip to drone.zip
Uploaded: drone.zip -> /content/drone.zip

Data extracted successfully.
Current files in DATA_DIR:
- drone-code-march-2026.pdf
- DJI_Product.zip


In [ ]:
# ============================================================
# 2. Imports and configuration
# ============================================================

from pathlib import Path
import os

# Project folders
BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "rag_data"
EXTRACT_DIR = BASE_DIR / "rag_extracted"
OUTPUT_DIR = BASE_DIR / "rag_outputs"

DATA_DIR.mkdir(exist_ok=True)
EXTRACT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# Automatically locate source files
# ------------------------------------------------------------

# Find the Drone Code PDF
pdf_candidates = list(DATA_DIR.rglob("*drone-code*.pdf"))

if len(pdf_candidates) == 0:
    raise FileNotFoundError("No Drone Code PDF found in rag_data. Please check the extracted files.")
else:
    DRONE_CODE_PDF = pdf_candidates[0]

# Find DJI product zip if it exists
dji_zip_candidates = list(DATA_DIR.rglob("*DJI*.zip")) + list(DATA_DIR.rglob("*Product*.zip"))

if len(dji_zip_candidates) > 0:
    DJI_ZIP = dji_zip_candidates[0]
else:
    DJI_ZIP = None

print("Detected files:")
print("Drone Code PDF:", DRONE_CODE_PDF)

if DJI_ZIP is not None:
    print("DJI product ZIP:", DJI_ZIP)
else:
    print("DJI product ZIP: not found. The product files may already be extracted inside rag_data.")

# ------------------------------------------------------------
# Model settings
# ------------------------------------------------------------

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Chunking settings
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

# Baseline retrieval setting
BASELINE_TOP_K = 5

# Enhanced retrieval settings
HYBRID_CANDIDATES = 25
ENHANCED_TOP_K = 6
HYBRID_ALPHA = 0.55

# Generation model
# A local open-source instruction model is used instead of an external API.
LOCAL_LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Generation model:", LOCAL_LLM_MODEL_NAME)

print("\nConfiguration completed.")

Detected files:
Drone Code PDF: /content/rag_data/drone-code-march-2026.pdf
DJI product ZIP: /content/rag_data/DJI_Product.zip
Generation model: Qwen/Qwen2.5-1.5B-Instruct

Configuration completed.


In [ ]:
# ============================================================
# 3. Document loading functions
# ============================================================

import re
import zipfile
from pathlib import Path
from typing import List, Dict, Any

import fitz  # PyMuPDF


def clean_text(text: str) -> str:
    """
    Clean extracted text by removing special characters, excessive whitespace,
    and simple page-number artefacts.
    """
    if not text:
        return ""

    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"Page\s+\d+\s+of\s+\d+", " ", text, flags=re.I)
    return text.strip()


def read_pdf(path: Path) -> List[Dict[str, Any]]:
    """
    Read a PDF file page by page and store each page as a document record.
    """
    records = []

    try:
        doc = fitz.open(path)

        for i, page in enumerate(doc, start=1):
            text = clean_text(page.get_text("text"))

            if text:
                records.append({
                    "source_file": path.name,
                    "source_path": str(path),
                    "page": i,
                    "text": text,
                    "doc_type": "pdf"
                })

        doc.close()

    except Exception as e:
        print(f"Warning: failed to read PDF {path}: {e}")

    return records


def read_text_file(path: Path) -> List[Dict[str, Any]]:
    """
    Read text-like files such as TXT, MD, HTML, CSV, and JSON.
    """
    records = []

    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
        text = clean_text(text)

        if text:
            records.append({
                "source_file": path.name,
                "source_path": str(path),
                "page": None,
                "text": text,
                "doc_type": path.suffix.lower().replace(".", "")
            })

    except Exception as e:
        print(f"Warning: failed to read text file {path}: {e}")

    return records


def extract_nested_zip_files(search_dir: Path, extract_dir: Path) -> None:
    """
    Find and extract zip files inside the data folder.
    This is useful when drone.zip contains another archive such as DJI_Product.zip.
    """
    zip_files = list(search_dir.rglob("*.zip"))

    if not zip_files:
        print("No nested zip files found.")
        return

    for zip_path in zip_files:
        target_dir = extract_dir / zip_path.stem
        target_dir.mkdir(parents=True, exist_ok=True)

        try:
            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(target_dir)

            print(f"Extracted nested zip: {zip_path.name} -> {target_dir}")

        except Exception as e:
            print(f"Warning: failed to extract {zip_path}: {e}")


def load_documents(data_dir: Path, extract_dir: Path) -> List[Dict[str, Any]]:
    """
    Load all supported documents from the extracted data folder.
    It reads PDFs and text-like files recursively. If nested ZIP files exist,
    they are extracted first and then included in the loading process.
    """
    all_records = []

    # Step 1: extract any zip files inside rag_data
    extract_nested_zip_files(data_dir, extract_dir)

    # Step 2: read from both rag_data and rag_extracted
    search_dirs = [data_dir, extract_dir]

    supported_text_extensions = {
        ".txt", ".md", ".html", ".htm", ".csv", ".json"
    }

    for folder in search_dirs:
        for path in folder.rglob("*"):
            if not path.is_file():
                continue

            suffix = path.suffix.lower()

            # Skip zip files after extraction
            if suffix == ".zip":
                continue

            if suffix == ".pdf":
                all_records.extend(read_pdf(path))

            elif suffix in supported_text_extensions:
                all_records.extend(read_text_file(path))

    print(f"\nLoaded {len(all_records)} document records.")

    if all_records:
        print("Example loaded record:")
        print("Source file:", all_records[0]["source_file"])
        print("Document type:", all_records[0]["doc_type"])
        print("Page:", all_records[0]["page"])
        print("Text preview:", all_records[0]["text"][:300])

    return all_records


documents = load_documents(DATA_DIR, EXTRACT_DIR)

Extracted nested zip: DJI_Product.zip -> /content/rag_extracted/DJI_Product

Loaded 169 document records.
Example loaded record:
Source file: drone-code-march-2026.pdf
Document type: pdf
Page: 1
Text preview: CAA | The Drone and Model Aircraft Code | CAP2320 | March 2026   The Drone and Model Aircraft Code > For flying drones, model aeroplanes, model gliders, model helicopters, and other unmanned aircraft outdoors in the Open Over People (A1) and Far from People (A3) sub-categories. > Follow this Code to


# B. Preprocessing
chunking, embedding, vector store.

## Chunking

In [ ]:
# ============================================================
# 4. Chunking
# ============================================================

from typing import List, Dict, Any


def split_text_into_chunks(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP
) -> List[str]:
    """
    Split a long text into overlapping chunks.

    chunk_size controls the approximate maximum number of characters per chunk.
    chunk_overlap keeps some repeated text between neighbouring chunks so that
    important context is less likely to be lost at chunk boundaries.
    """
    if not text:
        return []

    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        # Move forward, but keep some overlap with the previous chunk
        start = end - chunk_overlap

        # Safety check to avoid infinite loops
        if start < 0:
            start = 0

        if start >= text_length:
            break

    return chunks


def create_chunks(
    documents: List[Dict[str, Any]],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP
) -> List[Dict[str, Any]]:
    """
    Convert loaded document records into chunk records.

    Each chunk keeps metadata from the original document, such as source file,
    source path, page number, and document type. This metadata is important for
    traceability and grounded generation later.
    """
    chunk_records = []
    chunk_id = 0

    for doc_id, doc in enumerate(documents):
        text = doc.get("text", "")

        text_chunks = split_text_into_chunks(
            text=text,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        for local_chunk_id, chunk_text in enumerate(text_chunks):
            chunk_records.append({
                "chunk_id": chunk_id,
                "doc_id": doc_id,
                "local_chunk_id": local_chunk_id,
                "source_file": doc.get("source_file"),
                "source_path": doc.get("source_path"),
                "page": doc.get("page"),
                "doc_type": doc.get("doc_type"),
                "text": chunk_text
            })

            chunk_id += 1

    print(f"Created {len(chunk_records)} chunks from {len(documents)} document records.")

    if chunk_records:
        print("\nExample chunk:")
        print("Chunk ID:", chunk_records[0]["chunk_id"])
        print("Source file:", chunk_records[0]["source_file"])
        print("Page:", chunk_records[0]["page"])
        print("Text preview:", chunk_records[0]["text"][:500])

    return chunk_records


chunks = create_chunks(
    documents=documents,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

Created 434 chunks from 169 document records.

Example chunk:
Chunk ID: 0
Source file: drone-code-march-2026.pdf
Page: 1
Text preview: CAA | The Drone and Model Aircraft Code | CAP2320 | March 2026   The Drone and Model Aircraft Code > For flying drones, model aeroplanes, model gliders, model helicopters, and other unmanned aircraft outdoors in the Open Over People (A1) and Far from People (A3) sub-categories. > Follow this Code to make sure you always fly safely and legally. Many of the rules in the Drone and Model Aircraft Code are legal requirements, and if you disobey these rules, you are committing a criminal offence. > Th


## Embeddings and vector store

In [ ]:
# ============================================================
# 5. Embeddings and vector store
# ============================================================

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# 5.1 Load embedding model
# ------------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)


# ------------------------------------------------------------
# 5.2 Create embeddings for all chunks
# ------------------------------------------------------------

chunk_texts = [chunk["text"] for chunk in chunks]

print(f"Number of chunks to embed: {len(chunk_texts)}")

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

chunk_embeddings = chunk_embeddings.astype("float32")

print("Chunk embeddings created.")
print("Embedding matrix shape:", chunk_embeddings.shape)

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
Number of chunks to embed: 434


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Chunk embeddings created.
Embedding matrix shape: (434, 384)


In [ ]:
# ------------------------------------------------------------
# 5.3 Build FAISS vector store
# ------------------------------------------------------------

embedding_dim = chunk_embeddings.shape[1]

# Because embeddings are normalized, inner product is equivalent to cosine similarity
faiss_index = faiss.IndexFlatIP(embedding_dim)

faiss_index.add(chunk_embeddings)

print("FAISS vector store created.")
print("Number of vectors in FAISS index:", faiss_index.ntotal)
print("Embedding dimension:", embedding_dim)

FAISS vector store created.
Number of vectors in FAISS index: 434
Embedding dimension: 384


In [ ]:
# ------------------------------------------------------------
# 5.4 Test vector retrieval
# ------------------------------------------------------------

def vector_retrieve(query: str, top_k: int = 5):
    """
    Retrieve the most similar chunks using FAISS vector search.
    This function will be used as the baseline retrieval method.
    """
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        chunk = chunks[idx].copy()
        chunk["vector_score"] = float(score)
        results.append(chunk)

    return results


# Quick retrieval test
test_query = "What is the maximum height for flying a drone?"

test_results = vector_retrieve(test_query, top_k=5)

print("Test query:", test_query)
print("\nTop retrieved chunks:")

for i, result in enumerate(test_results, start=1):
    print(f"\n--- Result {i} ---")
    print("Source file:", result["source_file"])
    print("Page:", result["page"])
    print("Vector score:", round(result["vector_score"], 4))
    print("Text preview:", result["text"][:500])

Test query: What is the maximum height for flying a drone?

Top retrieved chunks:

--- Result 1 ---
Source file: drone-code-march-2026.pdf
Page: 14
Vector score: 0.7161
Text preview: CAA | The Drone and Model Aircraft Code | CAP2320 | March 2026   Where you can fly Fly below 120m (400ft) Flying below the legal height limit of 120m (400ft) will reduce the risk of coming across other aircraft, which normally fly higher than this. Always look and listen out for other aircraft that may be flying below 120m (400ft), such as air ambulances, police helicopters, and low-flying military aircraft. Never fly more than 120m (400ft) above the earth’s surface Flying where there are hills,

--- Result 2 ---
Source file: drone-code-march-2026.pdf
Page: 47
Vector score: 0.678
Text preview: CAA | The Drone and Model Aircraft Code | CAP2320 | March 2026   Less common flying Flying over very tall structures If the person or organisation responsible for a very tall structure over 105m asks you to carry out

# C. Two Pipeline

In [ ]:
# ============================================================
# 6. BM25 keyword index for hybrid retrieval
# ============================================================

import re
from typing import List
from rank_bm25 import BM25Okapi


def tokenize(text: str) -> List[str]:
    """
    Simple tokenizer for BM25 keyword retrieval.
    It lowercases text and keeps English words and numbers.
    """
    text = text.lower()
    return re.findall(r"[a-z0-9]+", text)


# Extract chunk texts from the chunk records
chunk_texts = [chunk["text"] for chunk in chunks]

# Tokenize all chunks for BM25
tokenized_corpus = [tokenize(text) for text in chunk_texts]

# Build BM25 keyword index
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 keyword index created.")
print("Number of indexed chunks:", len(tokenized_corpus))

BM25 keyword index created.
Number of indexed chunks: 434


In [ ]:
# ============================================================
# 7. Baseline retrieval: vector-only
# ============================================================

import pandas as pd


def baseline_retrieve(query: str, top_k: int = BASELINE_TOP_K) -> pd.DataFrame:
    """
    Baseline retrieval method.

    This function uses vector-only retrieval:
    query -> embedding -> FAISS search -> top-k chunks

    It does not use BM25 keyword retrieval or reranking.
    """
    # Convert the user query into an embedding
    q_emb = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # Search the FAISS vector index
    scores, indices = faiss_index.search(q_emb, top_k)

    results = []

    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        if idx == -1:
            continue

        chunk = chunks[idx].copy()

        results.append({
            "rank": rank,
            "chunk_id": chunk.get("chunk_id"),
            "source_file": chunk.get("source_file"),
            "page": chunk.get("page"),
            "doc_type": chunk.get("doc_type"),
            "vector_score": float(score),
            "text": chunk.get("text")
        })

    return pd.DataFrame(results)


# Quick test
baseline_results = baseline_retrieve(
    "What should a drone operator check before flying?",
    top_k=3
)

baseline_results

,rank,chunk_id,source_file,page,doc_type,vector_score,text
0,1,49,drone-code-march-2026.pdf,27,pdf,0.697927,CAA | The Drone and Model Aircraft Code | CAP2...
1,2,59,drone-code-march-2026.pdf,32,pdf,0.666665,CAA | The Drone and Model Aircraft Code | CAP2...
2,3,18,drone-code-march-2026.pdf,11,pdf,0.659037,CAA | The Drone and Model Aircraft Code | CAP2...


In [ ]:
# ============================================================
# 8. Enhanced retrieval: hybrid retrieval + explicit reranking
# ============================================================

import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder


# ------------------------------------------------------------
# 8.1 Load reranker model
# ------------------------------------------------------------

print("Loading reranker model...")

reranker = CrossEncoder(RERANKER_MODEL_NAME)

print("Reranker loaded:", RERANKER_MODEL_NAME)


# ------------------------------------------------------------
# 8.2 Utility: min-max normalization
# ------------------------------------------------------------

def minmax_normalize(x: np.ndarray) -> np.ndarray:
    """
    Normalize scores into the range [0, 1].
    This allows vector scores and BM25 scores to be combined.
    """
    x = np.asarray(x, dtype=float)

    if x.max() - x.min() < 1e-9:
        return np.zeros_like(x)

    return (x - x.min()) / (x.max() - x.min())


# ------------------------------------------------------------
# 8.3 Hybrid retrieval candidates
# ------------------------------------------------------------

def hybrid_retrieve_candidates(
    query: str,
    candidate_k: int = HYBRID_CANDIDATES,
    alpha: float = HYBRID_ALPHA
) -> pd.DataFrame:
    """
    Retrieve candidate chunks using hybrid retrieval.

    Hybrid retrieval combines:
    1. Dense vector similarity score
    2. BM25 keyword score

    alpha controls the weight:
    - alpha close to 1 gives more weight to vector retrieval
    - alpha close to 0 gives more weight to BM25 keyword retrieval
    """
    # Query embedding
    q_emb = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # Dense vector scores against all chunk embeddings
    vector_scores = (chunk_embeddings @ q_emb[0])

    # BM25 keyword scores against all tokenized chunks
    bm25_scores = np.array(
        bm25.get_scores(tokenize(query)),
        dtype=float
    )

    # Normalize both score types before combining them
    vector_norm = minmax_normalize(vector_scores)
    bm25_norm = minmax_normalize(bm25_scores)

    hybrid_scores = alpha * vector_norm + (1 - alpha) * bm25_norm

    # Select top candidate_k chunks by hybrid score
    top_indices = np.argsort(hybrid_scores)[::-1][:candidate_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        chunk = chunks[idx].copy()

        results.append({
            "candidate_rank": rank,
            "chunk_id": chunk.get("chunk_id"),
            "source_file": chunk.get("source_file"),
            "source_path": chunk.get("source_path"),
            "page": chunk.get("page"),
            "doc_type": chunk.get("doc_type"),
            "vector_score": float(vector_scores[idx]),
            "bm25_score": float(bm25_scores[idx]),
            "hybrid_score": float(hybrid_scores[idx]),
            "text": chunk.get("text")
        })

    return pd.DataFrame(results)


# ------------------------------------------------------------
# 8.4 Rerank hybrid candidates with CrossEncoder
# ------------------------------------------------------------

def enhanced_retrieve(
    query: str,
    candidate_k: int = HYBRID_CANDIDATES,
    top_k: int = ENHANCED_TOP_K,
    alpha: float = HYBRID_ALPHA
) -> pd.DataFrame:
    """
    Enhanced retrieval method.

    Step 1: retrieve candidate chunks using hybrid retrieval.
    Step 2: rerank the candidates using a cross-encoder reranker.
    Step 3: return the top-k reranked chunks.

    This is the enhanced retrieval pipeline used for the improved RAG system.
    """
    candidates = hybrid_retrieve_candidates(
        query=query,
        candidate_k=candidate_k,
        alpha=alpha
    )

    if candidates.empty:
        return candidates

    # CrossEncoder takes query-text pairs
    pairs = [
        [query, text]
        for text in candidates["text"].tolist()
    ]

    rerank_scores = reranker.predict(pairs)

    candidates = candidates.copy()
    candidates["rerank_score"] = rerank_scores

    # Sort by reranker score
    reranked = candidates.sort_values(
        by="rerank_score",
        ascending=False
    ).head(top_k)

    reranked = reranked.reset_index(drop=True)
    reranked["rank"] = range(1, len(reranked) + 1)

    return reranked[
        [
            "rank",
            "chunk_id",
            "source_file",
            "page",
            "doc_type",
            "vector_score",
            "bm25_score",
            "hybrid_score",
            "rerank_score",
            "text"
        ]
    ]


# ------------------------------------------------------------
# 8.5 Quick test
# ------------------------------------------------------------

test_query = "What should a drone operator check before flying?"

enhanced_results = enhanced_retrieve(
    query=test_query,
    candidate_k=HYBRID_CANDIDATES,
    top_k=3,
    alpha=HYBRID_ALPHA
)

enhanced_results

Loading reranker model...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


,rank,chunk_id,source_file,page,doc_type,vector_score,bm25_score,hybrid_score,rerank_score,text
0,1,80,drone-code-march-2026.pdf,44,pdf,0.624922,13.819663,0.858771,5.775423,CAA | The Drone and Model Aircraft Code | CAP2...
1,2,76,drone-code-march-2026.pdf,42,pdf,0.578933,11.654903,0.767206,5.725360,CAA | The Drone and Model Aircraft Code | CAP2...
2,3,75,drone-code-march-2026.pdf,41,pdf,0.613356,15.195969,0.886291,5.716584,CAA | The Drone and Model Aircraft Code | CAP2...


In [ ]:
# ============================================================
# 9. Prompt construction for baseline and enhanced pipelines
# ============================================================

import pandas as pd


def format_context(retrieved_df: pd.DataFrame) -> str:
    """
    Convert retrieved chunks into a readable context string for the LLM.

    Each retrieved chunk is formatted with its source metadata, including
    source file, page number if available, chunk ID, and retrieved text.
    """
    if retrieved_df is None or retrieved_df.empty:
        return "No relevant context was retrieved."

    blocks = []

    for _, row in retrieved_df.iterrows():
        source_file = row.get("source_file", "unknown source")
        chunk_id = row.get("chunk_id", "unknown chunk")

        # Page number may be missing for non-PDF files
        page = row.get("page", None)

        if pd.notna(page):
            try:
                page_info = f", page {int(page)}"
            except Exception:
                page_info = f", page {page}"
        else:
            page_info = ""

        source = f"{source_file}{page_info}"

        text = row.get("text", "")

        block = (
            f"[Source: {source} | chunk_id: {chunk_id}]\n"
            f"{text}"
        )

        blocks.append(block)

    return "\n\n---\n\n".join(blocks)


def build_baseline_prompt(query: str, retrieved_df: pd.DataFrame) -> str:
    """
    Build a simple baseline prompt.

    This prompt uses retrieved context, but it gives the model only a basic
    instruction. It is intentionally less strict than the enhanced grounded prompt.
    """
    context = format_context(retrieved_df)

    prompt = f"""
You are a helpful assistant. Use the following context to answer the user's question.

Context:
{context}

Question:
{query}

Answer:
"""

    return prompt.strip()


def build_enhanced_prompt(query: str, retrieved_df: pd.DataFrame) -> str:
    """
    Build a grounded prompt for the enhanced RAG pipeline.

    This prompt explicitly instructs the model to answer only from retrieved
    evidence and to avoid unsupported claims.
    """
    context = format_context(retrieved_df)

    prompt = f"""
You are a careful assistant answering questions about drone rules and DJI product documentation.

You must follow these rules:
1. Answer only using the retrieved context below.
2. Do not use outside knowledge.
3. If the retrieved context does not contain enough evidence, say:
   "The retrieved documents do not provide enough information to answer this question."
4. Do not invent product specifications, legal rules, dates, or numerical limits.
5. When possible, mention the relevant source file and page number.
6. Keep the answer clear, concise, and practical.

Retrieved context:
{context}

User question:
{query}

Grounded answer:
"""

    return prompt.strip()


# ------------------------------------------------------------
# Quick test
# ------------------------------------------------------------

test_query = "What should a drone operator check before flying?"

baseline_context = baseline_retrieve(test_query, top_k=3)
enhanced_context = enhanced_retrieve(test_query, top_k=3)

baseline_test_prompt = build_baseline_prompt(test_query, baseline_context)
enhanced_test_prompt = build_enhanced_prompt(test_query, enhanced_context)

print("Baseline prompt preview:")
print(baseline_test_prompt[:1500])

print("\n" + "=" * 80 + "\n")

print("Enhanced prompt preview:")
print(enhanced_test_prompt[:1500])

Baseline prompt preview:
You are a helpful assistant. Use the following context to answer the user's question.

Context:
[Source: drone-code-march-2026.pdf, page 27 | chunk_id: 49]
CAA | The Drone and Model Aircraft Code | CAP2320 | March 2026   Making every flight safe Make sure your drone or model aircraft is fit to fly Check fuel and battery levels Take special care to check that fuel and battery levels will last through your flight. This includes any extra fuel you might need in an emergency or for flying in difficult weather, such as windy conditions. Remember to check the battery power in the controller too. Check any built-in software is up to date The built-in software (called firmware) controls important navigation and flying controls. Depending on the type of drone or model aircraft you have, this could include: > how your drone uses its power > how your drone knows its position > how your drone lands if there’s a problem > in some cases, the latest information on flight rest

In [ ]:
# ============================================================
# 10. Local LLM generation function without API
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# ------------------------------------------------------------
# 10.1 Choose a local open-source model
# ------------------------------------------------------------

LOCAL_LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"


print("Loading local LLM model...")
print("Model:", LOCAL_LLM_MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_LLM_MODEL_NAME,
    trust_remote_code=True
)

local_llm = AutoModelForCausalLM.from_pretrained(
    LOCAL_LLM_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

print("Local LLM loaded successfully.")


# ------------------------------------------------------------
# 10.2 Local answer generation
# ------------------------------------------------------------

def generate_answer_local(
    prompt: str,
    max_new_tokens: int = 400,
    temperature: float = 0.2
) -> str:
    """
    Generate an answer using a local open-source LLM instead of an external API.
    This avoids Gemini API quota issues.
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    # Convert the prompt into the model's chat format
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(local_llm.device)

    with torch.no_grad():
        outputs = local_llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Remove the input prompt from the generated output
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

Loading local LLM model...
Model: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Local LLM loaded successfully.


In [ ]:
# ============================================================
# 10.3 Run baseline and enhanced pipelines with local LLM
# ============================================================

def run_baseline_pipeline(query: str) -> dict:
    """
    Run the baseline RAG pipeline:
    vector-only retrieval -> vanilla prompt -> local LLM answer
    """
    retrieved_df = baseline_retrieve(
        query=query,
        top_k=BASELINE_TOP_K
    )

    prompt = build_baseline_prompt(
        query=query,
        retrieved_df=retrieved_df
    )

    answer = generate_answer_local(prompt)

    return {
        "query": query,
        "pipeline": "baseline",
        "retrieved_context": retrieved_df,
        "prompt": prompt,
        "answer": answer
    }


def run_enhanced_pipeline(query: str) -> dict:
    """
    Run the enhanced RAG pipeline:
    hybrid retrieval -> explicit reranking -> grounded prompt -> local LLM answer
    """
    retrieved_df = enhanced_retrieve(
        query=query,
        candidate_k=HYBRID_CANDIDATES,
        top_k=ENHANCED_TOP_K,
        alpha=HYBRID_ALPHA
    )

    prompt = build_enhanced_prompt(
        query=query,
        retrieved_df=retrieved_df
    )

    answer = generate_answer_local(prompt)

    return {
        "query": query,
        "pipeline": "enhanced",
        "retrieved_context": retrieved_df,
        "prompt": prompt,
        "answer": answer
    }

In [ ]:
# ============================================================
# 10.4 Quick test
# ============================================================

test_query = "What should a drone operator check before flying?"

baseline_output = run_baseline_pipeline(test_query)
enhanced_output = run_enhanced_pipeline(test_query)

print("Baseline answer:")
print(baseline_output["answer"])

print("\n" + "=" * 80 + "\n")

print("Enhanced answer:")
print(enhanced_output["answer"])

Baseline answer:
A drone operator should check several things before flying:

1. Ensure their drone or model aircraft is fit to fly.
2. Check fuel and battery levels, including any additional fuel needed for emergencies or challenging weather conditions.
3. Verify the battery power in the controller.
4. Confirm that the built-in software is up-to-date, especially related to navigation, landing procedures, and flight restrictions.
5. Keep track of any flight restriction zones and other airspace restrictions.


Enhanced answer:
A drone operator should check the following items before flying:

1. **Get an Operator ID**: Ensure that the drone or model aircraft requires an Operator ID and obtain one if needed. Label the drone or model aircraft with the Operator ID.

2. **Label the Drone or Model Aircraft**: Make sure the Operator ID is clearly labeled on the drone or model aircraft. It should be visible from the outside or accessible through an easily opened compartment.

3. **Fly with Remo

# Test

## Query

In [ ]:
# ============================================================
# 11. Test set
# ============================================================

import pandas as pd

test_queries = [
    {
        "query_id": "Q1",
        "query_type": "ambiguous product selection / UK use",
        "question": "I want to use a DJI drone in the UK. Which models could I choose?",
        "expected_keywords": ["DJI", "drone", "weight", "camera", "flight"]
    },
    {
        "query_id": "Q2",
        "query_type": "product comparison",
        "question": "Which DJI drones are lighter and easier for casual users to carry?",
        "expected_keywords": ["DJI", "weight", "portable", "light", "mini"]
    },
    {
        "query_id": "Q3",
        "query_type": "regulation / simple fact",
        "question": "What does the UK Drone Code say about keeping a drone in sight?",
        "expected_keywords": ["sight", "visible", "line", "drone"]
    },
    {
        "query_id": "Q4",
        "query_type": "product specs + user need",
        "question": "I want a DJI drone mainly for photography and video. What product specifications should I compare?",
        "expected_keywords": ["camera", "video", "photo", "sensor", "resolution"]
    },
    {
        "query_id": "Q5",
        "query_type": "compliance / operator responsibility",
        "question": "Do I need an Operator ID before flying a DJI drone in the UK?",
        "expected_keywords": ["operator", "id", "label", "drone"]
    },
    {
        "query_id": "Q6",
        "query_type": "deep context / product and regulation",
        "question": "If I choose a heavier DJI drone, what UK flying rules or responsibilities should I pay attention to?",
        "expected_keywords": ["weight", "operator", "people", "safe", "drone"]
    },
    {
        "query_id": "Q7",
        "query_type": "ambiguous regulation query/ flying location",
        "question": "I want to fly my drone near a town park. What should I check before deciding whether the flight is allowed?",
        "expected_keywords": ["park", "people", "crowd", "restriction", "airspace", "permission", "safe"]
    },
    {
        "query_id": "Q8",
        "query_type": "edge case / missing information",
        "question": "Does the document explain how to repair a broken DJI drone motor?",
        "expected_keywords": ["repair", "motor", "not", "information"]
    },
    {
        "query_id": "Q9",
        "query_type": "pre-flight / safety checklist",
        "question": "What basic safety checks should a drone operator complete before flying?",
        "expected_keywords": ["check", "before", "fly", "safe", "battery"]
    }
]

test_set_df = pd.DataFrame(test_queries)
test_set_df

,query_id,query_type,question,expected_keywords
0,Q1,ambiguous product selection / UK use,I want to use a DJI drone in the UK. Which mod...,"[DJI, drone, weight, camera, flight]"
1,Q2,product comparison,Which DJI drones are lighter and easier for ca...,"[DJI, weight, portable, light, mini]"
2,Q3,regulation / simple fact,What does the UK Drone Code say about keeping ...,"[sight, visible, line, drone]"
3,Q4,product specs + user need,I want a DJI drone mainly for photography and ...,"[camera, video, photo, sensor, resolution]"
4,Q5,compliance / operator responsibility,Do I need an Operator ID before flying a DJI d...,"[operator, id, label, drone]"
5,Q6,deep context / product and regulation,"If I choose a heavier DJI drone, what UK flyin...","[weight, operator, people, safe, drone]"
6,Q7,ambiguous regulation query/ flying location,I want to fly my drone near a town park. What ...,"[park, people, crowd, restriction, airspace, p..."
7,Q8,edge case / missing information,Does the document explain how to repair a brok...,"[repair, motor, not, information]"
8,Q9,pre-flight / safety checklist,What basic safety checks should a drone operat...,"[check, before, fly, safe, battery]"


## retrieval

In [ ]:
# ============================================================
# 12. Retrieval evaluation
# ============================================================

from typing import List, Dict, Any


def chunk_matches_keywords(text: str, keywords: List[str]) -> bool:
    """
    Check whether a chunk contains at least one expected keyword.
    """
    text_l = str(text).lower()

    return any(
        keyword.lower() in text_l
        for keyword in keywords
    )


def keyword_hit_rate(retrieved_df: pd.DataFrame, expected_keywords: List[str]) -> Dict[str, Any]:
    """
    Calculate how many expected keywords appear in the retrieved chunks.

    This is a lightweight retrieval evaluation method for a small RAG assignment.
    """
    retrieved_text = " ".join(
        retrieved_df["text"].fillna("").astype(str).tolist()
    ).lower()

    matched_keywords = []

    for keyword in expected_keywords:
        if keyword.lower() in retrieved_text:
            matched_keywords.append(keyword)

    hit_rate = len(matched_keywords) / len(expected_keywords) if expected_keywords else 0

    return {
        "matched_keywords": matched_keywords,
        "num_expected_keywords": len(expected_keywords),
        "num_matched_keywords": len(matched_keywords),
        "keyword_hit_rate": hit_rate
    }


def evaluate_retrieval_for_query(
    query: str,
    expected_keywords: List[str],
    pipeline: str
) -> Dict[str, Any]:
    """
    Evaluate retrieval for one query under either the baseline or enhanced pipeline.
    """
    if pipeline == "baseline":
        retrieved = baseline_retrieve(
            query=query,
            top_k=BASELINE_TOP_K
        )

    elif pipeline == "enhanced":
        retrieved = enhanced_retrieve(
            query=query,
            candidate_k=HYBRID_CANDIDATES,
            top_k=ENHANCED_TOP_K,
            alpha=HYBRID_ALPHA
        )

    else:
        raise ValueError("pipeline must be either 'baseline' or 'enhanced'")

    eval_result = keyword_hit_rate(
        retrieved_df=retrieved,
        expected_keywords=expected_keywords
    )

    relevant_chunks = retrieved[
        retrieved["text"].apply(
            lambda x: chunk_matches_keywords(x, expected_keywords)
        )
    ]

    precision_at_k = len(relevant_chunks) / len(retrieved) if len(retrieved) > 0 else 0

    return {
        "pipeline": pipeline,
        "retrieved_df": retrieved,
        "matched_keywords": eval_result["matched_keywords"],
        "num_expected_keywords": eval_result["num_expected_keywords"],
        "num_matched_keywords": eval_result["num_matched_keywords"],
        "keyword_hit_rate": eval_result["keyword_hit_rate"],
        "precision_at_k": precision_at_k
    }


def evaluate_retrieval_test_set(test_queries: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Run retrieval evaluation for both baseline and enhanced pipelines.
    """
    rows = []

    for item in test_queries:
        query_id = item["query_id"]
        query_type = item["query_type"]
        question = item["question"]
        expected_keywords = item["expected_keywords"]

        for pipeline in ["baseline", "enhanced"]:
            result = evaluate_retrieval_for_query(
                query=question,
                expected_keywords=expected_keywords,
                pipeline=pipeline
            )

            retrieved_df = result["retrieved_df"]

            top_sources = ", ".join(
                retrieved_df["source_file"]
                .dropna()
                .astype(str)
                .unique()[:3]
            )

            top_chunk_ids = ", ".join(
                retrieved_df["chunk_id"]
                .dropna()
                .astype(str)
                .head(3)
                .tolist()
            )

            rows.append({
                "query_id": query_id,
                "query_type": query_type,
                "question": question,
                "pipeline": pipeline,
                "expected_keywords": ", ".join(expected_keywords),
                "matched_keywords": ", ".join(result["matched_keywords"]),
                "num_expected_keywords": result["num_expected_keywords"],
                "num_matched_keywords": result["num_matched_keywords"],
                "keyword_hit_rate": result["keyword_hit_rate"],
                "precision_at_k": result["precision_at_k"],
                "top_sources": top_sources,
                "top_chunk_ids": top_chunk_ids
            })

    return pd.DataFrame(rows)


retrieval_eval_df = evaluate_retrieval_test_set(test_queries)

retrieval_eval_df

,query_id,query_type,question,pipeline,expected_keywords,matched_keywords,num_expected_keywords,num_matched_keywords,keyword_hit_rate,precision_at_k,top_sources,top_chunk_ids
0,Q1,ambiguous product selection / UK use,I want to use a DJI drone in the UK. Which mod...,baseline,"DJI, drone, weight, camera, flight","DJI, drone, weight, camera",5,4,0.800000,1.000000,"DJI Mini 3 - Specs - DJI.pdf, drone-code-march...","155, 8, 256"
1,Q1,ambiguous product selection / UK use,I want to use a DJI drone in the UK. Which mod...,enhanced,"DJI, drone, weight, camera, flight","DJI, drone, weight, camera, flight",5,5,1.000000,1.000000,"drone-code-march-2026.pdf, DJI Lito X1 - Specs...","8, 16, 9"
2,Q2,product comparison,Which DJI drones are lighter and easier for ca...,baseline,"DJI, weight, portable, light, mini","DJI, weight, light, mini",5,4,0.800000,1.000000,"DJI Avata 2 - FPV Drone - Specs - DJI.pdf, DJI...","203, 148, 256"
3,Q2,product comparison,Which DJI drones are lighter and easier for ca...,enhanced,"DJI, weight, portable, light, mini","DJI, light, mini",5,3,0.600000,1.000000,"DJI Mini 3 - Specs - DJI.pdf, DJI Avata 2 - FP...","155, 193, 92"
4,Q3,regulation / simple fact,What does the UK Drone Code say about keeping ...,baseline,"sight, visible, line, drone","sight, line, drone",4,3,0.750000,1.000000,drone-code-march-2026.pdf,"18, 65, 87"
5,Q3,regulation / simple fact,What does the UK Drone Code say about keeping ...,enhanced,"sight, visible, line, drone","sight, drone",4,2,0.500000,1.000000,drone-code-march-2026.pdf,"20, 18, 17"
6,Q4,product specs + user need,I want a DJI drone mainly for photography and ...,baseline,"camera, video, photo, sensor, resolution","camera, video, photo",5,3,0.600000,1.000000,"DJI Neo 2 - Specs - DJI.pdf, DJI Mini 4K _ DJI...","138, 256, 193"
7,Q4,product specs + user need,I want a DJI drone mainly for photography and ...,enhanced,"camera, video, photo, sensor, resolution","camera, video, photo",5,3,0.600000,1.000000,"DJI Mini 3 - Specs - DJI.pdf, DJI Mini 4K _ DJ...","155, 256, 270"
8,Q5,compliance / operator responsibility,Do I need an Operator ID before flying a DJI d...,baseline,"operator, id, label, drone","operator, id, label, drone",4,4,1.000000,1.000000,drone-code-march-2026.pdf,"10, 5, 75"
9,Q5,compliance / operator responsibility,Do I need an Operator ID before flying a DJI d...,enhanced,"operator, id, label, drone","operator, id, label, drone",4,4,1.000000,1.000000,drone-code-march-2026.pdf,"5, 78, 73"


In [ ]:
# ============================================================
# 13. Retrieval process log
# ============================================================

def create_retrieval_process_log(
    test_queries: List[Dict[str, Any]],
    top_n: int = 3
) -> pd.DataFrame:
    """
    Create a detailed retrieval log showing the top retrieved chunks
    for both baseline and enhanced pipelines.
    """
    rows = []

    for item in test_queries:
        query_id = item["query_id"]
        query_type = item["query_type"]
        question = item["question"]

        baseline_results = baseline_retrieve(
            query=question,
            top_k=top_n
        )

        enhanced_results = enhanced_retrieve(
            query=question,
            candidate_k=HYBRID_CANDIDATES,
            top_k=top_n,
            alpha=HYBRID_ALPHA
        )

        for _, row in baseline_results.iterrows():
            rows.append({
                "query_id": query_id,
                "query_type": query_type,
                "question": question,
                "pipeline": "baseline",
                "rank": row.get("rank"),
                "chunk_id": row.get("chunk_id"),
                "source_file": row.get("source_file"),
                "page": row.get("page"),
                "score_summary": f"vector_score={row.get('vector_score'):.4f}",
                "text_preview": str(row.get("text", ""))[:500]
            })

        for _, row in enhanced_results.iterrows():
            rows.append({
                "query_id": query_id,
                "query_type": query_type,
                "question": question,
                "pipeline": "enhanced",
                "rank": row.get("rank"),
                "chunk_id": row.get("chunk_id"),
                "source_file": row.get("source_file"),
                "page": row.get("page"),
                "score_summary": (
                    f"vector_score={row.get('vector_score'):.4f}; "
                    f"bm25_score={row.get('bm25_score'):.4f}; "
                    f"hybrid_score={row.get('hybrid_score'):.4f}; "
                    f"rerank_score={row.get('rerank_score'):.4f}"
                ),
                "text_preview": str(row.get("text", ""))[:500]
            })

    return pd.DataFrame(rows)


retrieval_process_log_df = create_retrieval_process_log(
    test_queries=test_queries,
    top_n=3
)

retrieval_process_log_df

,query_id,query_type,question,pipeline,rank,chunk_id,source_file,page,score_summary,text_preview
0,Q1,ambiguous product selection / UK use,I want to use a DJI drone in the UK. Which mod...,baseline,1,155,DJI Mini 3 - Specs - DJI.pdf,1,vector_score=0.7193,elected countries and regions. DJI Mini 3 Acce...
1,Q1,ambiguous product selection / UK use,I want to use a DJI drone in the UK. Which mod...,baseline,2,8,drone-code-march-2026.pdf,5,vector_score=0.6658,. You can still fly your drone or model aircra...
2,Q1,ambiguous product selection / UK use,I want to use a DJI drone in the UK. Which mod...,baseline,3,256,DJI Mini 4K _ DJI Mini 2 SE - Specs - DJI.pdf,1,vector_score=0.6434,"n and sea-level altitude, with the aircraft co..."
3,Q1,ambiguous product selection / UK use,I want to use a DJI drone in the UK. Which mod...,enhanced,1,8,drone-code-march-2026.pdf,5,vector_score=0.6658; bm25_score=8.2607; hybrid...,. You can still fly your drone or model aircra...
4,Q1,ambiguous product selection / UK use,I want to use a DJI drone in the UK. Which mod...,enhanced,2,16,drone-code-march-2026.pdf,9,vector_score=0.5161; bm25_score=20.1834; hybri...,CAA | The Drone and Model Aircraft Code | CAP2...
5,Q1,ambiguous product selection / UK use,I want to use a DJI drone in the UK. Which mod...,enhanced,3,9,drone-code-march-2026.pdf,6,vector_score=0.5286; bm25_score=11.7585; hybri...,CAA | The Drone and Model Aircraft Code | CAP2...
6,Q2,product comparison,Which DJI drones are lighter and easier for ca...,baseline,1,203,DJI Avata 2 - FPV Drone - Specs - DJI.pdf,6,vector_score=0.6066,Battery Capacity 2600 mAh Weight Approx. 240 g...
7,Q2,product comparison,Which DJI drones are lighter and easier for ca...,baseline,2,148,DJI Neo 2 - Specs - DJI.pdf,7,vector_score=0.5896,Weight Approx. 320 g Dimensions 104.2×150×45.2...
8,Q2,product comparison,Which DJI drones are lighter and easier for ca...,baseline,3,256,DJI Mini 4K _ DJI Mini 2 SE - Specs - DJI.pdf,1,vector_score=0.5894,"n and sea-level altitude, with the aircraft co..."
9,Q2,product comparison,Which DJI drones are lighter and easier for ca...,enhanced,1,155,DJI Mini 3 - Specs - DJI.pdf,1,vector_score=0.5561; bm25_score=8.5664; hybrid...,elected countries and regions. DJI Mini 3 Acce...


In [ ]:
# ============================================================
# 14. Save retrieval evaluation outputs
# ============================================================

retrieval_eval_path = OUTPUT_DIR / "retrieval_evaluation_results.csv"
retrieval_log_path = OUTPUT_DIR / "retrieval_process_log.csv"

retrieval_eval_df.to_csv(retrieval_eval_path, index=False)
retrieval_process_log_df.to_csv(retrieval_log_path, index=False)

print("Saved retrieval evaluation results to:")
print(retrieval_eval_path)

print("\nSaved retrieval process log to:")
print(retrieval_log_path)

Saved retrieval evaluation results to:
/content/rag_outputs/retrieval_evaluation_results.csv

Saved retrieval process log to:
/content/rag_outputs/retrieval_process_log.csv


In [ ]:
# ============================================================
# Download retrieval output files in Google Colab
# ============================================================

from google.colab import files

files.download("/content/rag_outputs/retrieval_evaluation_results.csv")
files.download("/content/rag_outputs/retrieval_process_log.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## generation

In [ ]:
# ============================================================
# 15. Generation demo
# ============================================================

import pandas as pd


# Select a small subset for generation demo, because local LLM generation can be slow
generation_demo_queries = [
    item for item in test_queries
    if item["query_id"] in ["Q1", "Q5", "Q7"]
]


def summarise_retrieved_context(retrieved_df: pd.DataFrame) -> str:
    """
    Create a compact summary of retrieved sources and chunk IDs.
    This helps show what evidence was used before generation.
    """
    if retrieved_df is None or retrieved_df.empty:
        return "No retrieved context."

    summary_parts = []

    for _, row in retrieved_df.iterrows():
        source_file = row.get("source_file", "unknown source")
        page = row.get("page", None)
        chunk_id = row.get("chunk_id", "unknown")

        if pd.notna(page):
            source = f"{source_file}, page {page}, chunk_id {chunk_id}"
        else:
            source = f"{source_file}, chunk_id {chunk_id}"

        summary_parts.append(source)

    return " | ".join(summary_parts)


def run_generation_demo(test_items: list) -> pd.DataFrame:
    """
    Run both baseline and enhanced pipelines and save generated answers.
    """
    rows = []

    for item in test_items:
        query_id = item["query_id"]
        query_type = item["query_type"]
        question = item["question"]

        print("=" * 100)
        print(f"Running generation for {query_id}: {question}")
        print("=" * 100)

        # -------------------------
        # Baseline pipeline
        # -------------------------
        baseline_output = run_baseline_pipeline(question)
        baseline_retrieved = baseline_output["retrieved_context"]
        baseline_answer = baseline_output["answer"]

        rows.append({
            "query_id": query_id,
            "query_type": query_type,
            "question": question,
            "pipeline": "baseline",
            "retrieved_sources": summarise_retrieved_context(baseline_retrieved),
            "answer": baseline_answer
        })

        print("\nBaseline answer:")
        print(baseline_answer)

        # -------------------------
        # Enhanced pipeline
        # -------------------------
        enhanced_output = run_enhanced_pipeline(question)

        enhanced_retrieved = enhanced_output["retrieved_context"]
        enhanced_answer = enhanced_output["answer"]

        rows.append({
            "query_id": query_id,
            "query_type": query_type,
            "question": question,
            "pipeline": "enhanced",
            "retrieved_sources": summarise_retrieved_context(enhanced_retrieved),
            "answer": enhanced_answer
        })

        print("\nEnhanced answer:")
        print(enhanced_answer)
        print("\n")

    return pd.DataFrame(rows)


generation_demo_df = run_generation_demo(generation_demo_queries)

generation_demo_df

Running generation for Q1: I want to use a DJI drone in the UK. Which models could I choose?

Baseline answer:
Based on the information provided, there is no specific mention of any particular DJI drone models approved for use in the UK. The sources only provide general specifications and FAQs about drones in general, without specifying which models are allowed in the UK. To determine which models are suitable for use in the UK, you would need to check the official DJI website or contact their customer support directly. They will have the most up-to-date and accurate information regarding drone regulations and permitted models within the UK.

Enhanced answer:
To determine which DJI models you can use in the UK, we need to consider the regulatory requirements specified in the provided context:

1. **Class Marks**:
   - **C class**: These meet European class standards but do not specify UK class standards. You can fly a C class drone as if it's the corresponding UK class drone until 31 D

,query_id,query_type,question,pipeline,retrieved_sources,answer
0,Q1,ambiguous product selection / UK use,I want to use a DJI drone in the UK. Which mod...,baseline,"DJI Mini 3 - Specs - DJI.pdf, page 1, chunk_id...","Based on the information provided, there is no..."
1,Q1,ambiguous product selection / UK use,I want to use a DJI drone in the UK. Which mod...,enhanced,"drone-code-march-2026.pdf, page 5, chunk_id 8 ...",To determine which DJI models you can use in t...
2,Q5,compliance / operator responsibility,Do I need an Operator ID before flying a DJI d...,baseline,"drone-code-march-2026.pdf, page 6, chunk_id 10...","Yes, according to the provided information, yo..."
3,Q5,compliance / operator responsibility,Do I need an Operator ID before flying a DJI d...,enhanced,"drone-code-march-2026.pdf, page 4, chunk_id 5 ...","Yes, according to the retrieved context, you n..."
4,Q7,ambiguous regulation query/ flying location,I want to fly my drone near a town park. What ...,baseline,"drone-code-march-2026.pdf, page 11, chunk_id 1...",Before deciding whether to fly your drone near...
5,Q7,ambiguous regulation query/ flying location,I want to fly my drone near a town park. What ...,enhanced,"drone-code-march-2026.pdf, page 22, chunk_id 3...",To determine if flying your drone near a town ...


In [ ]:
# ============================================================
# Export generation results
# ============================================================

import pandas as pd
import shutil
from pathlib import Path

# Make sure OUTPUT_DIR exists
OUTPUT_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 1. Check whether generation_demo_df exists
# ------------------------------------------------------------

if "generation_demo_df" not in globals():
    raise NameError(
        "generation_demo_df was not found. "
        "Please run the generation demo section before exporting results."
    )

if generation_demo_df.empty:
    raise ValueError(
        "generation_demo_df is empty. "
        "Please check whether generation results were created successfully."
    )

print("Generation demo results found.")
print("Number of rows:", len(generation_demo_df))


# ------------------------------------------------------------
# 2. Export full generation demo log
# ------------------------------------------------------------

generation_csv_path = OUTPUT_DIR / "generation_demo_log.csv"
generation_xlsx_path = OUTPUT_DIR / "generation_demo_log.xlsx"

generation_demo_df.to_csv(
    generation_csv_path,
    index=False,
    encoding="utf-8-sig"
)

generation_demo_df.to_excel(
    generation_xlsx_path,
    index=False
)

print("Saved full generation demo log:")
print(generation_csv_path)
print(generation_xlsx_path)


# ------------------------------------------------------------
# 3. Create a shorter demo log for submission
# ------------------------------------------------------------

# Keep only the most useful columns if they exist
preferred_columns = [
    "query_id",
    "query_type",
    "question",
    "pipeline",
    "retrieved_sources",
    "answer"
]

available_columns = [
    col for col in preferred_columns
    if col in generation_demo_df.columns
]

short_demo_log_df = generation_demo_df[available_columns].copy()

# Add a commentary column for manual notes
# You can fill this in after reading the baseline/enhanced outputs.
short_demo_log_df["commentary"] = ""

short_demo_csv_path = OUTPUT_DIR / "short_demo_log.csv"
short_demo_xlsx_path = OUTPUT_DIR / "short_demo_log.xlsx"

short_demo_log_df.to_csv(
    short_demo_csv_path,
    index=False,
    encoding="utf-8-sig"
)

short_demo_log_df.to_excel(
    short_demo_xlsx_path,
    index=False
)

print("\nSaved short demo log for submission:")
print(short_demo_csv_path)
print(short_demo_xlsx_path)


# ------------------------------------------------------------
# 4. Zip all output files for downloading
# ------------------------------------------------------------

zip_base = str(OUTPUT_DIR.parent / "rag_outputs")
zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=OUTPUT_DIR
)

print("\nAll output files zipped:")
print(zip_path)

Generation demo results found.
Number of rows: 6
Saved full generation demo log:
/content/rag_outputs/generation_demo_log.csv
/content/rag_outputs/generation_demo_log.xlsx

Saved short demo log for submission:
/content/rag_outputs/short_demo_log.csv
/content/rag_outputs/short_demo_log.xlsx

All output files zipped:
/content/rag_outputs.zip


In [ ]:
# ============================================================
# Download output zip in Google Colab
# ============================================================

try:
    from google.colab import files
    files.download("/content/rag_outputs.zip")
except Exception:
    print("Not running in Google Colab. Please find the zip file in your project folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>